# Multimodal Medical Assistant - Google Colab Setup

This notebook sets up and runs the Medical Assistant with GPU acceleration on Google Colab.

## Prerequisites
- Enable GPU: Runtime → Change runtime type → T4 GPU
- Colab automatically has most dependencies pre-installed

## Step 1: Check GPU Availability

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ GPU not enabled. Go to Runtime → Change runtime type → T4 GPU")

## Step 2: Clone Repository

In [ ]:
# Clone your repository (replace with your repo URL)
!git clone https://github.com/YOUR_USERNAME/Medical_AI_Agent.git
%cd Medical_AI_Agent/Codebase

## Step 3: Install Dependencies

In [ ]:
# Install required packages
!pip install -q -r requirements.txt

# Install Ollama (for Llama 3)
!curl -fsSL https://ollama.com/install.sh | sh

print("✓ Dependencies installed")

## Step 4: Start Ollama Service

In [ ]:
import subprocess
import time

# Start Ollama in background
ollama_process = subprocess.Popen(['ollama', 'serve'], 
                                   stdout=subprocess.PIPE, 
                                   stderr=subprocess.PIPE)
time.sleep(5)  # Wait for service to start

# Pull Llama 3 model
!ollama pull llama3.2

print("✓ Ollama service started")
print("✓ Llama 3.2 model ready")

## Step 5: Test Device Configuration

In [ ]:
# Run device test script
!python test_cuda_devices.py

## Option A: Launch Gradio UI (Recommended for Colab)

Gradio automatically creates a public URL in Colab!

In [ ]:
# Launch Gradio UI
!python main.py --ui

# Gradio will automatically display a public URL like:
# Running on public URL: https://xxxxx.gradio.live

## Option B: CLI Mode (Interactive Commands)

In [ ]:
# Import the orchestrator
from cli_main import MedicalAssistantOrchestrator

# Initialize
orchestrator = MedicalAssistantOrchestrator()
print("✓ Medical Assistant initialized")
print("Use orchestrator.analyze_text(), .analyze_image(), or .multimodal_fusion()")

### Example: Analyze Text (Clinical Report)

In [ ]:
from models import MedicalText

# Sample clinical text
clinical_text = """
Patient: 68-year-old male
Chief Complaint: Cough, fever, and shortness of breath for 4 days
History: COPD, Type 2 Diabetes, Former smoker
Vitals: BP 145/88, HR 98, Temp 38.5C, SpO2 92% on room air
Labs: WBC 16,800, CRP elevated
"""

# Analyze
result = orchestrator.analyze_text(
    MedicalText(text=clinical_text, text_type="clinical_note")
)

print("Analysis Result:")
print(result.analysis)

### Example: Analyze Image (X-ray)

In [ ]:
from models import MedicalImage, ImageModality
from google.colab import files

# Upload an image
print("Upload a medical image (X-ray, CT, MRI):")
uploaded = files.upload()

# Get the uploaded filename
image_path = list(uploaded.keys())[0]

# Analyze
result = orchestrator.analyze_image(
    MedicalImage(
        image_path=image_path,
        modality=ImageModality.XRAY,
        body_part="chest"
    )
)

print("\nImage Analysis Result:")
print(result.findings)

### Example: Multimodal Fusion (Text + Image)

In [ ]:
# Multimodal analysis (combines text and image)
result = orchestrator.multimodal_fusion(
    medical_text=MedicalText(text=clinical_text, text_type="clinical_note"),
    medical_image=MedicalImage(
        image_path=image_path,
        modality=ImageModality.XRAY,
        body_part="chest"
    )
)

print("\nMultimodal Analysis Result:")
print(result.integrated_assessment)

## Option C: Test BiomedCLIP with Sample Image

In [ ]:
# Test BiomedCLIP on sample image
!python test_biomedclip.py

## Performance Monitoring

In [ ]:
# Check GPU memory usage
!nvidia-smi

# Check model cache
!du -sh ~/.cache/huggingface/hub/

## Tips for Colab

1. **GPU Runtime**: Always enable T4 GPU for faster inference
2. **Session Timeout**: Colab sessions timeout after 90 minutes of inactivity
3. **Gradio Public URL**: Automatically generated, lasts for 72 hours
4. **File Uploads**: Use `files.upload()` to upload medical images
5. **Downloads**: Use `files.download(filename)` to save results

## Expected Performance (T4 GPU)

- BioBERT embedding: ~10-20ms
- BiomedCLIP image analysis: ~0.5-1s (if GPU enabled)
- Llama 3 generation: ~1-2s
- Total pipeline: ~3-5s

## Troubleshooting

**Issue: CUDA out of memory**
- Solution: Runtime → Factory reset runtime

**Issue: Ollama not responding**
- Solution: Restart Ollama cell (Step 4)

**Issue: Gradio not showing URL**
- Solution: Check if port 7860 is in use, restart runtime
